# Homework 1 · Embed your own knowledge base
(Embeddings, dimension cost)

1. Write 10 short sentences about a topic you know well (a hobby, a
city, a sport). Give each one a topic label.
2. Embed them with embed_documents() and print the number of
vectors and the dimension of each.
3. Calculate how much memory 5 million of your vectors would need
in float32 (dim × 4 bytes).
4. Request the same text with dimensions=256 from openai/
text-embedding-3-large. How much memory do you save,
and what do you give up?
Goal: see that an embedding is just a fixed-length list of floats, and that
its size drives storage cost.

In [15]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b:free"

def embed_documents(texts: list[str], dims: int=2048) -> list[list[float]]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts, dimensions=dims)
    return [emb.embedding for emb in response.data]

coffee_knowledge_base = {
    "Beans": "Coffee beans grow on green bushes in warm countries.",
    "Color": "Dark roasted beans are shinier and taste stronger.",
    "Caffeine": "A cup of morning coffee helps people wake up.",
    "Grind": "People crush whole beans into powder before brewing.",
    "Water": "Hot water passes through the coffee grounds to extract flavor.",
    "Milk": "Many people add warm milk and sugar to sweeten their drink.",
    "Filter": "Paper filters keep the coffee grounds out of the cup.",
    "Ice": "Iced coffee is a cold, refreshing drink for hot summer days.",
    "Café": "People often meet friends to chat at local coffee shops.",
    "Decaf": "Decaffeinated coffee removes almost all of the caffeine."
}

labels = list(coffee_knowledge_base.keys())
documents = list(coffee_knowledge_base.values())

doc_embeddings = embed_documents(documents,2048)
dims = len(doc_embeddings[0])
print("Number of embeddings: ", len(doc_embeddings))
print("Dimensions of each vector: ", dims)

no_of_docs = 5_000_000
bytes_per_float = 4
total_bytes = no_of_docs * dims * bytes_per_float
total_gb = total_bytes/1024**3
print(f"\nMemory for 5M vectors (dim={dims}, float32): {total_gb:.2f} GB")

Number of embeddings:  10
Dimensions of each vector:  2048

Memory for 5M vectors (dim=2048, float32): 38.15 GB


# Homework 2 · Compare distance metrics (Cosine,
Euclidean, Dot product)
1. Embed three queries of your own: one that closely matches a
document, one that is loosely related, and one that is completely
unrelated.
2. For each query, compute cosine similarity, Euclidean distance and
dot product against all your documents (cosine_similarity,
euclidean_distances, linear_kernel).
3. Print the top 3 documents per metric. Do the rankings agree?
4. Check whether your vectors are unit length with
np.linalg.norm. Explain why the rankings do or do not match,
using d² = 2 − 2·cos.
Goal: understand when the three metrics agree, and why normalized
vectors make them equivalent.


In [37]:
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances,
    linear_kernel,
    manhattan_distances,
)

doc_vectors = np.array(doc_embeddings)

queries = {
    "Closely Matching": "How do you grind coffee beans?",
    "Loosely Related": "What is a good morning beverage to wake me up?",
    "Completely Unrelated": "What is the capital city of Australia?",
}

query_names = list(queries.keys())
query_texts = list(queries.values())

query_embeddings = embed_documents(query_texts)
query_vectors = np.array(query_embeddings)

# -------------------------------------------------------------
# Part 2 & 3: Compute metrics and print top 3 documents
# -------------------------------------------------------------
for q_name, q_vec in zip(queries.keys(), query_vectors):
    q_vec_2d = q_vec.reshape(1, -1)

    # Compute metrics against all documents
    cos_sim = cosine_similarity(q_vec_2d, doc_vectors)[0]
    euc_dist = euclidean_distances(q_vec_2d, doc_vectors)[0]
    l1_dist = manhattan_distances(q_vec_2d, doc_vectors)[0]
    dot_prod = linear_kernel(q_vec_2d, doc_vectors)[0]

    # Get top 3 indices for each metric
    # Cosine & Dot Product: Higher is better
    top_cos = np.argsort(cos_sim)[::-1][:3]
    # Euclidean: Lower distance is better
    top_euc = np.argsort(euc_dist)[:3]
    top_dot = np.argsort(dot_prod)[::-1][:3]
    top_l1 = np.argsort(l1_dist)[::-1][:3]

    print(f"\n==============================")
    print(f"QUERY ({q_name}): '{queries[q_name]}'")
    print(f"==============================")

    print("\nTop 3 by Cosine Similarity:")
    for idx in top_cos:
        print(f"  - [{labels[idx]}] (Score: {cos_sim[idx]:.4f}): {documents[idx]}")

    print("\nTop 3 by Dot Product:")
    for idx in top_dot:
        print(f"  - [{labels[idx]}] (Score: {dot_prod[idx]:.4f}): {documents[idx]}")

    print("\nTop 3 by Euclidean(L2) Distance:")
    for idx in top_euc:
        print(f"  - [{labels[idx]}] (Distance: {euc_dist[idx]:.4f}): {documents[idx]}")

    print("\nTop 3 by Manhattan(L1) Distance:")
    for idx in top_l1:
        print(f"  - [{labels[idx]}] (Score: {dot_prod[idx]:.4f}): {documents[idx]}")


# -------------------------------------------------------------
# Part 4: Check unit length and explain rankings
# -------------------------------------------------------------
doc_norms = np.linalg.norm(doc_vectors, axis=1)
query_norms = np.linalg.norm(query_vectors, axis=1)

print("\n------------------------------")
print("Part 4: Vector Normalization Check")
print("------------------------------")
print(f"Are all document vectors unit length? {np.allclose(doc_norms, 1.0)}")
print(f"Are all query vectors unit length? {np.allclose(query_norms, 1.0)}")
print(f"Sample document norms: {doc_norms[:3]}")


QUERY (Closely Matching): 'How do you grind coffee beans?'

Top 3 by Cosine Similarity:
  - [Grind] (Score: 0.5432): People crush whole beans into powder before brewing.
  - [Water] (Score: 0.4336): Hot water passes through the coffee grounds to extract flavor.
  - [Beans] (Score: 0.3911): Coffee beans grow on green bushes in warm countries.

Top 3 by Dot Product:
  - [Grind] (Score: 0.5432): People crush whole beans into powder before brewing.
  - [Water] (Score: 0.4336): Hot water passes through the coffee grounds to extract flavor.
  - [Beans] (Score: 0.3911): Coffee beans grow on green bushes in warm countries.

Top 3 by Euclidean(L2) Distance:
  - [Grind] (Distance: 0.9558): People crush whole beans into powder before brewing.
  - [Water] (Distance: 1.0643): Hot water passes through the coffee grounds to extract flavor.
  - [Beans] (Distance: 1.1036): Coffee beans grow on green bushes in warm countries.

Top 3 by Manhattan(L1) Distance:
  - [Milk] (Score: 0.1166): Many people add

# Homework 3 · Build your first Qdrant collection
(Collection, points, search)
1. Create a new collection with the correct size and
Distance.COSINE.
2. Upsert your documents as points with a payload of text and
topic.
3. Search with your three queries from Homework 2 and print score,
id and text.
4. Re-upsert one point with the same id but different text. Confirm
the count did not grow (qdrant.count(...)), and explain what
"upsert" did.
5. Check that Qdrant's ranking matches your manual retrieve()
function.
Goal: get comfortable with the collection → point → payload model and
verify Qdrant returns what the maths predicts

In [49]:
from qdrant_client import QdrantClient, models

# Initialize clients
# For local persistent storage, or use QdrantClient(":memory:") for ephemeral testing
qdrant = QdrantClient(":memory:")

vector_size = len(doc_embeddings[0])

collection_name = "coffee_knowledge"

# -------------------------------------------------------------
# Part 1: Create a new collection
# -------------------------------------------------------------
if qdrant.collection_exists(collection_name):
    qdrant.delete_collection(collection_name)

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size, distance=models.Distance.COSINE
    ),
)
print(f"Collection '{collection_name}' created successfully.")

# -------------------------------------------------------------
# Part 2: Upsert your documents as points with payloads
# -------------------------------------------------------------
points = []
for idx, (label, text) in enumerate(coffee_knowledge_base.items()):
    points.append(
        models.PointStruct(
            id=idx,  # using integer IDs (0 to 9)
            vector=doc_embeddings[idx],
            payload={"topic": label, "text": text},
        )
    )

qdrant.upsert(collection_name=collection_name, points=points)
print(f"Upserted {len(points)} points into Qdrant.")

# -------------------------------------------------------------
# Part 3: Search with your three queries from Homework 2
# -------------------------------------------------------------

print("\n--- Qdrant Search Results ---")
for q_name, q_vec in zip(queries.keys(), query_embeddings):
    search_results = qdrant.query_points(
        collection_name=collection_name, query=q_vec, limit=3
    ).points

    print(f"\nQuery ({q_name}): '{queries[q_name]}'")
    for res in search_results:
        print(
            f"  - ID: {res.id}, Score: {res.score:.4f}, Topic: [{res.payload['topic']}], Text: {res.payload['text']}"
        )


# -------------------------------------------------------------
# Part 4: Re-upsert one point and explain "upsert"
# -------------------------------------------------------------
target_id = 0
original_count = qdrant.count(collection_name=collection_name).count

# Re-upsert the same point ID (0) with modified text
updated_text = (
    "Updated: Coffee plants flourish in tropical, high-altitude regions."
)
updated_vector = embed_documents([updated_text])

qdrant.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=target_id,
            vector=updated_vector[0],
            payload={"topic": "Beans (Updated)", "text": updated_text},
        )
    ],
)

new_count = qdrant.count(collection_name=collection_name).count
print(f"\n--- Part 4: Upsert Check ---")
print(f"Count before re-upsert: {original_count}")
print(f"Count after re-upsert: {new_count}")
print(
    "Explanation: 'Upsert' stands for 'Update + Insert'. If a point ID already exists in the collection, it overwrites/updates that existing point rather than creating a duplicate. Therefore, the total point count remains unchanged."
)


# -------------------------------------------------------------
# Part 5: Check that Qdrant's ranking matches manual retrieve()
# -------------------------------------------------------------
# Manual cosine similarity check using scikit-learn against document vectors
test_q_vec = np.array(query_embeddings[0]).reshape(1, -1)
doc_mat = np.array(doc_embeddings)
# Point 0 was replaced above; compare against the vectors currently in Qdrant.
doc_mat[target_id] = updated_vector[0]
manual_scores = cosine_similarity(test_q_vec, doc_mat)[0]
manual_top_idx = np.argsort(manual_scores)[::-1][:3]

# Qdrant top results for the first query
qdrant_results = qdrant.query_points(
    collection_name=collection_name, query=query_embeddings[0], limit=3
).points
qdrant_top_ids = [res.id for res in qdrant_results]

print(f"\n--- Part 5: Ranking Validation ---")
print(f"Manual top indices (Cosine similarity): {manual_top_idx}")
print(f"Qdrant top point IDs: {qdrant_top_ids}")
print(
    f"Do the rankings match? {list(manual_top_idx[:len(qdrant_top_ids)]) == qdrant_top_ids}"
)

Collection 'coffee_knowledge' created successfully.
Upserted 10 points into Qdrant.

--- Qdrant Search Results ---

Query (Closely Matching): 'How do you grind coffee beans?'
  - ID: 3, Score: 0.5432, Topic: [Grind], Text: People crush whole beans into powder before brewing.
  - ID: 4, Score: 0.4336, Topic: [Water], Text: Hot water passes through the coffee grounds to extract flavor.
  - ID: 0, Score: 0.3911, Topic: [Beans], Text: Coffee beans grow on green bushes in warm countries.

Query (Loosely Related): 'What is a good morning beverage to wake me up?'
  - ID: 2, Score: 0.6850, Topic: [Caffeine], Text: A cup of morning coffee helps people wake up.
  - ID: 7, Score: 0.2973, Topic: [Ice], Text: Iced coffee is a cold, refreshing drink for hot summer days.
  - ID: 5, Score: 0.2911, Topic: [Milk], Text: Many people add warm milk and sugar to sweeten their drink.

Query (Completely Unrelated): 'What is the capital city of Australia?'
  - ID: 0, Score: 0.1026, Topic: [Beans], Text: Coffee

# Homework 4 · Exact vs HNSW and Recall@K (Search
strategies)
1. Duplicate your documents with small text variations until you have
at least 200 points. Embed them in batches.
2. Create two collections: one searched with exact=True, one with
an HNSW index (m=8 and m=32, ef_construct=100).
3. For 10 queries, compute Recall@5 of each HNSW setup against
the exact results while varying hnsw_ef in [8, 16, 32, 64,
128].
4. Record latency for each setting and plot recall vs latency (a simple
table is fine).
5. Write two sentences: which m / hnsw_ef would you pick for
production, and why?
Goal: treat exact search as ground truth, measure what approximate
search loses, and tune the recall/latency trade-off.

In [50]:
import time
import numpy as np
from qdrant_client import QdrantClient, models

# -------------------------------------------------------------
# Part 1: Expand Knowledge Base to 200+ points with variations & embed in batches
# -------------------------------------------------------------
base_coffee_data = {
    "Beans": "Coffee beans grow on green bushes in warm countries.",
    "Color": "Dark roasted beans are shinier and taste stronger.",
    "Caffeine": "A cup of morning coffee helps people wake up.",
    "Grind": "People crush whole beans into powder before brewing.",
    "Water": "Hot water passes through the coffee grounds to extract flavor.",
    "Milk": "Many people add warm milk and sugar to sweeten their drink.",
    "Filter": "Paper filters keep the coffee grounds out of the cup.",
    "Ice": "Iced coffee is a cold, refreshing drink for hot summer days.",
    "Café": "People often meet friends to chat at local coffee shops.",
    "Decaf": "Decaffeinated coffee removes almost all of the caffeine.",
}

# Duplicate and create variations to hit 200+ points
expanded_docs = []
modifiers = [
    "Always check freshness.",
    "Brewing takes patience.",
    "A popular choice.",
    "Enjoy your cup.",
    "Sourced carefully.",
    "Best served hot.",
    "A daily ritual.",
    "Roasted to perfection.",
    "Expertly crafted.",
    "Savor every sip.",
    "Rich and aromatic.",
    "Freshly prepared.",
    "Pure indulgence.",
    "Hand-picked selection.",
    "Timeless classic.",
    "Crafted with care.",
    "A delightful treat.",
    "Rich in flavor.",
    "The ultimate brew.",
    "A perfect morning start.",
]

doc_id = 0
for label, text in base_coffee_data.items():
    for i, mod in enumerate(modifiers):
        expanded_docs.append(
            {
                "id": doc_id,
                "topic": label,
                "text": f"{text} ({mod} var {i})",
            }
        )
        doc_id += 1

print(f"Total documents created: {len(expanded_docs)}")

# Embed in batches (e.g., batch size of 50 to respect API limits)
batch_size = 50
all_embeddings = []

for i in range(0, len(expanded_docs), batch_size):
    batch = expanded_docs[i : i + batch_size]
    texts = [doc["text"] for doc in batch]
    response = client.embeddings.create(
        model=EMBEDDING_MODEL, input=texts, dimensions=2048
    )
    all_embeddings.extend([emb.embedding for emb in response.data])

vector_size = len(all_embeddings[0])

# -------------------------------------------------------------
# Part 2: Create collections (Exact vs HNSW setups)
# -------------------------------------------------------------
qdrant = QdrantClient(":memory:")

exact_collection = "coffee_exact"
hnsw_collection_8 = "coffee_hnsw_m8"
hnsw_collection_32 = "coffee_hnsw_m32"

# 1. Exact Collection (HNSW disabled via on_disk or explicit config, or search-time exact=True)
qdrant.create_collection(
    collection_name=exact_collection,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE,
        on_disk=False,
    ),
)

# 2. HNSW Collection with m=8
qdrant.create_collection(
    collection_name=hnsw_collection_8,
    vectors_config=models.VectorParams(
        size=vector_size, distance=models.Distance.COSINE
    ),
    hnsw_config=models.HnswConfigDiff(m=8, ef_construct=100),
)

# 3. HNSW Collection with m=32
qdrant.create_collection(
    collection_name=hnsw_collection_32,
    vectors_config=models.VectorParams(
        size=vector_size, distance=models.Distance.COSINE
    ),
    hnsw_config=models.HnswConfigDiff(m=32, ef_construct=100),
)

# Prepare points
points = [
    models.PointStruct(
        id=doc["id"],
        vector=all_embeddings[idx],
        payload={"topic": doc["topic"], "text": doc["text"]},
    )
    for idx, doc in enumerate(expanded_docs)
]

# Upsert into all three collections
qdrant.upsert(collection_name=exact_collection, points=points)
qdrant.upsert(collection_name=hnsw_collection_8, points=points)
qdrant.upsert(collection_name=hnsw_collection_32, points=points)
print("Collections created and populated successfully.")

# -------------------------------------------------------------
# Part 3 & 4: Compute Recall@5 and Latency for 10 Queries across settings
# -------------------------------------------------------------
test_queries = [
    "How do you grind coffee beans?",
    "What is a good morning beverage to wake me up?",
    "Tell me about dark roasted coffee.",
    "How does water extract flavor?",
    "What are the benefits of paper filters?",
    "How is iced coffee prepared?",
    "Where do people meet for coffee?",
    "What does decaf coffee do?",
    "Are coffee beans grown in warm countries?",
    "How do you add milk and sugar?",
]

# Embed test queries
query_res = client.embeddings.create(
    model=EMBEDDING_MODEL, input=test_queries, dimensions=2048
)
query_vectors = [emb.embedding for emb in query_res.data]

ef_values = [8, 16, 32, 64, 128]
results_table = []


def calculate_recall_at_k(exact_ids, approx_ids, k=5):
    exact_set = set(exact_ids[:k])
    approx_set = set(approx_ids[:k])
    return len(exact_set.intersection(approx_set)) / k


# Run benchmarks
for m_val, coll_name in [
    (8, hnsw_collection_8),
    (32, hnsw_collection_32),
]:
    for ef in ef_values:
        total_recall = 0
        latencies = []

        for q_vec in query_vectors:
            # Get Ground Truth (Exact search)
            start_time = time.perf_counter()
            exact_res = qdrant.query_points(
                collection_name=exact_collection,
                query=q_vec,
                limit=5,
                search_params=models.SearchParams(exact=True),
            ).points
            _ = time.perf_counter() - start_time  # exact latency baseline

            exact_ids = [res.id for res in exact_res]

            # Get HNSW approximate results with measured latency
            start_time = time.perf_counter()
            approx_res = qdrant.query_points(
                collection_name=coll_name,
                query=q_vec,
                limit=5,
                search_params=models.SearchParams(
                    hnsw_ef=ef
                ),  # Pass search-time ef
            ).points
            latency = (time.perf_counter() - start_time) * 1000  # in ms
            latencies.append(latency)

            approx_ids = [res.id for res in approx_res]
            total_recall += calculate_recall_at_k(exact_ids, approx_ids, k=5)

        avg_recall = total_recall / len(test_queries)
        avg_latency = sum(latencies) / len(latencies)
        results_table.append(
            {"m": m_val, "ef": ef, "recall": avg_recall, "latency_ms": avg_latency}
        )

# Print Summary Table
print(
    f"\n{'m':<5} | {'search_ef':<10} | {'Recall@5':<10} | {'Avg Latency (ms)':<15}"
)
print("-" * 50)
for row in results_table:
    print(
        f"{row['m']:<5} | {row['ef']:<10} | {row['recall']:<10.2f} | {row['latency_ms']:<15.4f}"
    )

Total documents created: 200
Collections created and populated successfully.


C:\Users\Beher\AppData\Local\Temp\ipykernel_8860\2970545038.py:171: UserWarning: Local mode performs exact (brute-force) search, so `search_params` has no effect, with the exception of `idf`.
  exact_res = qdrant.query_points(



m     | search_ef  | Recall@5   | Avg Latency (ms)
--------------------------------------------------
8     | 8          | 1.00       | 3.2918         
8     | 16         | 1.00       | 3.5450         
8     | 32         | 1.00       | 3.5245         
8     | 64         | 1.00       | 4.1708         
8     | 128        | 1.00       | 3.3881         
32    | 8          | 1.00       | 3.2342         
32    | 16         | 1.00       | 3.1296         
32    | 32         | 1.00       | 3.6356         
32    | 64         | 1.00       | 3.2907         
32    | 128        | 1.00       | 3.5188         


# Homework 5 · Quantization and filtering (Compression,payload filters, final comparison)
1. Create an INT8 scalar-quantized collection and try
always_ram=True and always_ram=False. Compare recall
against the exact collection.
2. Add a level payload field if you do not have one, and create
payload indexes on topic and level.
3. Run the same query with: no filter, topic == "ai", and topic
== "ai" and level != "beginner" (must + must_not).
Show how the results change.
4. Build one summary table for your query set with the columns
Strategy · Recall@5 · Avg latency · Memory estimate and the
rows Exact, HNSW, HNSW + INT8, HNSW + filter.
5. In three bullet points, recommend a strategy for (a) 10 thousand
documents, (b) 100 million documents, (c) a multi-tenant SaaS
product.
Goal: bring every concept together and justify a search strategy with
measurements rather than guesses.
Stretch goal: wrap your best configuration in a
rag_retrieve(question) function and test it on five questions
you wrote yourself

In [52]:
import os
import time
from dotenv import load_dotenv
import numpy as np
from openai import OpenAI
from qdrant_client import QdrantClient, models
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances,
    linear_kernel,
    manhattan_distances,
)

# Initialize environment and client
load_dotenv()
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b:free"


def embed_documents(texts: list[str], dims: int = 2048) -> list[list[float]]:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL, input=texts, dimensions=dims
    )
    return [emb.embedding for emb in response.data]


# =============================================================
# HOMEWORK 1: Embed your own knowledge base
# =============================================================
coffee_knowledge_base = {
    "Beans": "Coffee beans grow on green bushes in warm countries.",
    "Color": "Dark roasted beans are shinier and taste stronger.",
    "Caffeine": "A cup of morning coffee helps people wake up.",
    "Grind": "People crush whole beans into powder before brewing.",
    "Water": "Hot water passes through the coffee grounds to extract flavor.",
    "Milk": "Many people add warm milk and sugar to sweeten their drink.",
    "Filter": "Paper filters keep the coffee grounds out of the cup.",
    "Ice": "Iced coffee is a cold, refreshing drink for hot summer days.",
    "Café": "People often meet friends to chat at local coffee shops.",
    "Decaf": "Decaffeinated coffee removes almost all of the caffeine.",
}

labels = list(coffee_knowledge_base.keys())
documents = list(coffee_knowledge_base.values())

doc_embeddings = embed_documents(documents, 2048)
dims = len(doc_embeddings[0])
print(f"Number of embeddings: {len(doc_embeddings)}")
print(f"Dimensions of each vector: {dims}")

no_of_docs = 5_000_000
bytes_per_float = 4
total_bytes = no_of_docs * dims * bytes_per_float
total_gb = total_bytes / 1024**3
print(f"Memory for 5M vectors (dim={dims}, float32): {total_gb:.2f} GB")


# =============================================================
# HOMEWORK 2: Compare distance metrics
# =============================================================
doc_vectors = np.array(doc_embeddings)

queries = {
    "Closely Matching": "How do you grind coffee beans?",
    "Loosely Related": "What is a good morning beverage to wake me up?",
    "Completely Unrelated": "What is the capital city of Australia?",
}

query_names = list(queries.keys())
query_texts = list(queries.values())
query_embeddings = embed_documents(query_texts, 2048)
query_vectors = np.array(query_embeddings)

print(
    f"\n{'query':<20} {'cosine sim':<12} {'dot product':<12} {'l2_dist':<12} {'l1_dist':<12}"
)
for q_name, q_vec in zip(query_texts, query_embeddings):
    q_vec_2d = np.array(q_vec).reshape(1, -1)
    cosine_sim = cosine_similarity(q_vec_2d, doc_vectors)[0]
    dot_product = linear_kernel(q_vec_2d, doc_vectors)[0]
    l2 = euclidean_distances(q_vec_2d, doc_vectors)[0]
    l1 = manhattan_distances(q_vec_2d, doc_vectors)[0]

    best_idx = np.argmax(cosine_sim)
    print(
        f"{q_name[:18]:<20} {cosine_sim[best_idx]:<12.4f} {dot_product[best_idx]:<12.4f} {l2[best_idx]:<12.4f} {l1[best_idx]:<12.4f}"
    )


# =============================================================
# HOMEWORK 3: Build your first Qdrant collection
# =============================================================
qdrant = QdrantClient(":memory:")
collection_name = "coffee_knowledge"

if qdrant.collection_exists(collection_name):
    qdrant.delete_collection(collection_name)

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=dims, distance=models.Distance.COSINE
    ),
)

points = [
    models.PointStruct(
        id=idx,
        vector=doc_embeddings[idx],
        payload={"topic": label, "text": text},
    )
    for idx, (label, text) in enumerate(coffee_knowledge_base.items())
]
qdrant.upsert(collection_name=collection_name, points=points)

# Re-upsert check
original_count = qdrant.count(collection_name=collection_name).count
updated_text = (
    "Updated: Coffee plants flourish in tropical, high-altitude regions."
)
updated_vector = embed_documents([updated_text], 2048)[0]
qdrant.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=0,
            vector=updated_vector,
            payload={"topic": "Beans (Updated)", "text": updated_text},
        )
    ],
)
new_count = qdrant.count(collection_name=collection_name).count
print(
    f"\n[HW3 Upsert Check] Count before: {original_count} | Count after: {new_count}"
)


# =============================================================
# HOMEWORK 4: Exact vs HNSW and Recall@K
# =============================================================
expanded_docs = []
modifiers = [
    "Always check freshness.",
    "Brewing takes patience.",
    "A popular choice.",
    "Enjoy your cup.",
    "Sourced carefully.",
    "Best served hot.",
    "A daily ritual.",
    "Roasted to perfection.",
    "Expertly crafted.",
    "Savor every sip.",
]

doc_id = 0
for label, text in coffee_knowledge_base.items():
    for i, mod in enumerate(modifiers):
        expanded_docs.append(
            {
                "id": doc_id,
                "topic": label,
                "text": f"{text} ({mod} var {i})",
            }
        )
        doc_id += 1

# Batch embedding generation for 100+ points
batch_size = 50
all_embeddings = []
for i in range(0, len(expanded_docs), batch_size):
    batch = expanded_docs[i : i + batch_size]
    texts = [doc["text"] for doc in batch]
    all_embeddings.extend(embed_documents(texts, 2048))

exact_col = "coffee_exact"
hnsw_col_8 = "coffee_hnsw_m8"
hnsw_col_32 = "coffee_hnsw_m32"

qdrant.create_collection(
    collection_name=exact_col,
    vectors_config=models.VectorParams(
        size=dims, distance=models.Distance.COSINE
    ),
)
qdrant.create_collection(
    collection_name=hnsw_col_8,
    vectors_config=models.VectorParams(
        size=dims, distance=models.Distance.COSINE
    ),
    hnsw_config=models.HnswConfigDiff(m=8, ef_construct=100),
)
qdrant.create_collection(
    collection_name=hnsw_col_32,
    vectors_config=models.VectorParams(
        size=dims, distance=models.Distance.COSINE
    ),
    hnsw_config=models.HnswConfigDiff(m=32, ef_construct=100),
)

expanded_points = [
    models.PointStruct(
        id=doc["id"],
        vector=all_embeddings[idx],
        payload={
            "topic": doc["topic"],
            "text": doc["text"],
            "level": (
                "beginner"
                if idx % 3 == 0
                else ("intermediate" if idx % 3 == 1 else "advanced")
            ),
        },
    )
    for idx, doc in enumerate(expanded_docs)
]

qdrant.upsert(collection_name=exact_col, points=expanded_points)
qdrant.upsert(collection_name=hnsw_col_8, points=expanded_points)
qdrant.upsert(collection_name=hnsw_col_32, points=expanded_points)


# =============================================================
# HOMEWORK 5: Quantization, payload filters, final comparison
# =============================================================
quant_col = "coffee_int8_ram"
qdrant.create_collection(
    collection_name=quant_col,
    vectors_config=models.VectorParams(
        size=dims, distance=models.Distance.COSINE
    ),
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8, always_ram=True
        )
    ),
)
qdrant.upsert(collection_name=quant_col, points=expanded_points)

filter_col = "coffee_filtered"
qdrant.create_collection(
    collection_name=filter_col,
    vectors_config=models.VectorParams(
        size=dims, distance=models.Distance.COSINE
    ),
    hnsw_config=models.HnswConfigDiff(m=32, ef_construct=100),
)
qdrant.upsert(collection_name=filter_col, points=expanded_points)

# Create payload indexes
qdrant.create_payload_index(
    collection_name=filter_col,
    field_name="topic",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
qdrant.create_payload_index(
    collection_name=filter_col,
    field_name="level",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

# Test payload filters safely
sample_q_vec = query_vectors[0]
res_nofilter = qdrant.query_points(
    collection_name=filter_col, query=sample_q_vec, limit=3
).points
res_topic = qdrant.query_points(
    collection_name=filter_col,
    query=sample_q_vec,
    limit=3,
    query_filter=models.Filter(
        must=[models.FieldCondition(key="topic", match=models.MatchValue(value="Beans"))]
    ),
).points
res_complex = qdrant.query_points(
    collection_name=filter_col,
    query=sample_q_vec,
    limit=3,
    query_filter=models.Filter(
        must=[models.FieldCondition(key="topic", match=models.MatchValue(value="Beans"))],
        must_not=[
            models.FieldCondition(
                key="level", match=models.MatchValue(value="beginner")
            )
        ],
    ),
).points

print("\n--- HW5 Filter Test Results ---")
print(
    f"No filter top result: [{res_nofilter[0].payload['topic']} | {res_nofilter[0].payload['level']}]"
    if res_nofilter
    else "No results"
)
print(
    f"Topic=='Beans' top result: [{res_topic[0].payload['topic']} | {res_topic[0].payload['level']}]"
    if res_topic
    else "No results"
)
print(
    f"Topic=='Beans' & level!='beginner': [{res_complex[0].payload['topic']} | {res_complex[0].payload['level']}]"
    if res_complex
    else "No results"
)

print("\nPipeline execution complete successfully!")

Number of embeddings: 10
Dimensions of each vector: 2048
Memory for 5M vectors (dim=2048, float32): 38.15 GB

query                cosine sim   dot product  l2_dist      l1_dist     
How do you grind c   0.5432       0.5432       0.9558       34.6578     
What is a good mor   0.6850       0.6850       0.7937       28.5453     
What is the capita   0.1026       0.1026       1.3397       48.6670     

[HW3 Upsert Check] Count before: 10 | Count after: 10

--- HW5 Filter Test Results ---
No filter top result: [Grind | intermediate]
Topic=='Beans' top result: [Beans | beginner]
Topic=='Beans' & level!='beginner': [Beans | intermediate]

Pipeline execution complete successfully!
